# Train Qwen Pruner on Colab

This notebook trains only the pruner using the saved provenance samples in `dataset/samples/`.
It assumes the repo is available on Google Drive and runs well on a single A100.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install torch torchvision transformers accelerate bitsandbytes pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 34.3 MB/s eta 0:00:00


In [ ]:
!pip install qwen-vl-utils
!pip install git+https://github.com/huggingface/transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 73.5 MB/s eta 0:00:00
  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-_w_7wpsz
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-_w_7wpsz
  Resolved https://github.com/huggingface/transformers to commit c6c7e189846ccaa3bb410569bfc6556d193c638d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.8.0.dev0-py3-none-any.whl size=11701580 sha256=f92f7628132a66cb0287cc6b613d81d8bfd86226a25cc2e84aa3a1ba643fae4a
  Stored in directory: /tmp/pip-ephem-wheel-cache-lbxqe81b/wheels/49/a7/50/c9fdabbf10e51bb1256adb0c1a587fedd7184f5bad28d47fe3
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successf

In [ ]:
import bitsandbytes as bnb
print(bnb.__version__)

0.49.2


In [ ]:
import torch
import sys
import os

# 1. Add the directory containing your script to the Python path
repo_dir = "/content/drive/MyDrive/dynamic_pruning/DynamicPruning_Final"
if repo_dir not in sys.path:
    sys.path.append(repo_dir)

sys.argv = [
    "train_pruner_qwen.py",
    "--samples-dirs",
    "/content/drive/MyDrive/dynamic_pruning/DynamicPruning_Final/execution_outputs_part1",
    "/content/drive/MyDrive/dynamic_pruning/DynamicPruning_Final/execution_outputs_part2",
    "--output-dir", "/content/drive/MyDrive/dynamic_pruning/DynamicPruning_Final/outputs",
    "--feature-cache-dir", "/content/drive/MyDrive/dynamic_pruning/DynamicPruning_Final/feature_cache",
    "--resume",
    "--model-id", "Qwen/Qwen2.5-VL-7B-Instruct",
    "--keep-ratio", "0.5",
    "--target-layer", "10",
    "--epochs", "50",
    "--lr", "0.0001",
    "--load-in-4bit",
    "--trust-remote-code",
]


In [ ]:
from pathlib import Path
from train_pruner_qwen import (
    parse_args,
    set_seed,
    resolve_device,
    resolve_model_dtype,
    load_sample_records,
)
args = parse_args()
set_seed(args.seed)

device = resolve_device()
model_dtype = resolve_model_dtype(device)

output_dir = Path(args.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

records = load_sample_records(args.samples_dirs)

Loaded 7500 samples from /content/drive/MyDrive/dynamic_pruning/DynamicPruning_Final/execution_outputs_part1
Loaded 7500 samples from /content/drive/MyDrive/dynamic_pruning/DynamicPruning_Final/execution_outputs_part2
Total loaded: 15000 samples


In [ ]:
from transformers import AutoProcessor, BitsAndBytesConfig

from transformers import Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

In [ ]:
import transformers
print(transformers.__version__)

5.8.0.dev0


In [ ]:
from train_pruner_qwen import (
    read_checkpoint,
    split_records
)

 # Resume or fresh start
start_epoch = 1
best_val_overlap = float("-inf")
history = []
train_indices = None
val_indices = None

checkpoint = read_checkpoint(output_dir, device) if args.resume else None
train_indices = checkpoint.get("train_indices") if checkpoint else None
val_indices = checkpoint.get("val_indices") if checkpoint else None

if train_indices is not None:
    index_map = {r["question_id"]: r for r in records}
    train_records = [index_map[i] for i in train_indices if i in index_map]
    val_records = [index_map[i] for i in val_indices if i in index_map]
    print(f"Restored split: {len(train_records)} train | {len(val_records)} val")
else:
    train_records, val_records = split_records(
        records=records,
        train_ratio=args.train_split,
        seed=args.seed,
        max_train_samples=args.max_train_samples,
        max_val_samples=args.max_val_samples,
    )
    print(f"Train samples: {len(train_records)} | Val samples: {len(val_records)}")


Train samples: 13500 | Val samples: 1500


In [ ]:
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from pruner import QueryAwarePruner
from train_pruner_qwen import (
    build_model_kwargs,
    freeze_non_pruner
)

In [ ]:
processor = AutoProcessor.from_pretrained(args.model_id)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    args.model_id,
    **build_model_kwargs(args, model_dtype),
).to(device)
freeze_non_pruner(model)

pruner_dim = model.model.language_model.embed_tokens.embedding_dim
pruner = QueryAwarePruner(
    dim=pruner_dim,
    num_heads=args.num_heads,
    use_multi_head=not args.single_head,
).to(device=device, dtype=torch.float32)

# model.pruner = pruner


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

In [ ]:
from train_pruner_qwen import load_checkpoint

optimizer = torch.optim.AdamW(
        pruner.parameters(),
        lr=args.lr,
        weight_decay=args.weight_decay,
    )

if checkpoint is None:
    start_epoch, best_val_overlap, history, _, _ = 1, float("-inf"), [], None, None
else:
    start_epoch, best_val_overlap, history, _, _ = load_checkpoint(checkpoint, pruner, optimizer, device)

In [ ]:
import importlib
import train_pruner_qwen
importlib.reload(train_pruner_qwen)
from train_pruner_qwen import ProvenanceFeatureDataset

train_dataset = ProvenanceFeatureDataset(
    records=train_records,
    model=model,
    processor=processor,
    device=device,
    keep_ratio=args.keep_ratio,
    target_layer=args.target_layer,
    feature_cache_dir=Path(args.feature_cache_dir) / "train" if args.feature_cache_dir else None,
)
val_dataset = ProvenanceFeatureDataset(
    records=val_records,
    model=model,
    processor=processor,
    device=device,
    keep_ratio=args.keep_ratio,
    target_layer=args.target_layer,
    feature_cache_dir=Path(args.feature_cache_dir) / "val" if args.feature_cache_dir else None,
)

Preparing records: 100%|██████████| 1500/1500 [00:49<00:00, 30.51it/s]


In [ ]:
train_dataset.materialize_missing_cache()
val_dataset.materialize_missing_cache()

Caching frozen Qwen features: 100%|██████████| 1498/1498 [16:48<00:00,  1.49it/s]


In [ ]:
from train_pruner_qwen import (
    run_epoch,
    save_checkpoint
)

In [ ]:
for epoch in range(start_epoch, args.epochs + 1):
    print(f"\nEpoch {epoch}/{args.epochs}")
    train_metrics = run_epoch(
        dataset=train_dataset,
        model=model,
        pruner=pruner,
        optimizer=optimizer,
        device=device,
        soft_target_weight=args.soft_target_weight,
        train=True,
    )
    val_metrics = run_epoch(
        dataset=val_dataset,
        model=model,
        pruner=pruner,
        optimizer=optimizer,
        device=device,
        soft_target_weight=args.soft_target_weight,
        train=False,
    )

    epoch_metrics = {"epoch": epoch, "train": train_metrics, "val": val_metrics}
    history.append(epoch_metrics)

    print(
        f"train loss={train_metrics['loss']:.4f} overlap={train_metrics['topk_overlap']:.4f} | "
        f"val loss={val_metrics['loss']:.4f} overlap={val_metrics['topk_overlap']:.4f}"
    )

    if val_metrics["topk_overlap"] > best_val_overlap:
        best_val_overlap = val_metrics["topk_overlap"]
        save_checkpoint(
            output_dir, epoch, pruner, optimizer, args,
            best_val_overlap, history, train_records, val_records
        )

    torch.save({
        "epoch": epoch,
        "best_val_topk_overlap": best_val_overlap,
        "state_dict": pruner.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "train_indices": [r["question_id"] for r in train_records],
        "val_indices": [r["question_id"] for r in val_records],
    }, output_dir / "last_pruner.pt")


Epoch 1/50


eval epoch: 100%|██████████| 1500/1500 [00:06<00:00, 220.08it/s, loss=0.3117, overlap=0.9109]


train loss=0.3400 overlap=0.8811 | val loss=0.3070 overlap=0.8970

Epoch 2/50


eval epoch: 100%|██████████| 1500/1500 [00:07<00:00, 211.30it/s, loss=0.2807, overlap=0.9154]


train loss=0.2875 overlap=0.9068 | val loss=0.2794 overlap=0.9112

Epoch 3/50


...

train loss=0.1110 overlap=0.9862 | val loss=0.2970 overlap=0.9499

Epoch 49/50


eval epoch: 100%|██████████| 1500/1500 [00:06<00:00, 217.74it/s, loss=0.3072, overlap=0.9625]


train loss=0.1105 overlap=0.9865 | val loss=0.3029 overlap=0.9490

Epoch 50/50


eval epoch: 100%|██████████| 1500/1500 [00:06<00:00, 215.76it/s, loss=0.3154, overlap=0.9523]


train loss=0.1099 overlap=0.9868 | val loss=0.3144 overlap=0.9491


In [ ]:
import json

summary = {
        "model_id": args.model_id,
        "samples_dir": args.samples_dirs,
        "num_train_samples": len(train_dataset),
        "num_val_samples": len(val_dataset),
        "keep_ratio": args.keep_ratio,
        "target_layer": args.target_layer,
        "best_val_topk_overlap": best_val_overlap,
        "history": history,
    }
(output_dir / "metrics.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"Saved best pruner checkpoint to {output_dir / 'best_pruner.pt'}")
print(f"Saved metrics to {output_dir / 'metrics.json'}")

Saved best pruner checkpoint to /content/drive/MyDrive/dynamic_pruning/DynamicPruning_Final/outputs/best_pruner.pt
Saved metrics to /content/drive/MyDrive/dynamic_pruning/DynamicPruning_Final/outputs/metrics.json
